# V7 Optuna retune on Kaggle P100

Sweeps `num_leaves`, `learning_rate`, `feature_fraction`, `bagging_fraction`, `min_data_in_leaf`, and `reg_lambda` for the V6 regression stage (pinball, `alpha=0.6`, target `target_qty_imputed`). Uses 4 rolling origins per trial as the selection objective (mean + 0.5 * std of WAPE).

Artefacts saved under `/kaggle/working`:
- `v7_optuna_best_params.json`
- `v7_optuna_study.pkl`
- `v7_optuna_top10.csv`


In [ ]:
import shutil, subprocess, sys
if shutil.which('nvidia-smi'):
    print(subprocess.check_output(['nvidia-smi', '-L']).decode())
else:
    print('CPU-only kernel')
print(sys.version)

In [ ]:
!pip install -q lightgbm==4.6.0 joblib pandas pyarrow matplotlib scikit-learn optuna==4.1.0
import os, shutil, sys, zipfile
INPUT_DIR = '/kaggle/input'; REPO_DIR = '/kaggle/working/repo'
os.makedirs(REPO_DIR, exist_ok=True)
print('kaggle/input contents:', os.listdir(INPUT_DIR))
candidates = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(f'{INPUT_DIR}/{d}')]
assert candidates, 'no dataset attached'
DATASET = (next((d for d in candidates if 'bpm' in d.lower() or 'v6' in d.lower()), None) or candidates[0])
DS_DIR = f'{INPUT_DIR}/{DATASET}'
print('using dataset', DS_DIR)

def _locate(name):
    if os.path.isdir(f'{DS_DIR}/{name}'): return f'{DS_DIR}/{name}'
    for root, dirs, _ in os.walk(DS_DIR):
        if name in dirs: return os.path.join(root, name)
    return None

for sub in ('src', 'scripts'):
    loc = _locate(sub)
    assert loc, f'missing {sub}/'
    shutil.copytree(loc, f'{REPO_DIR}/{sub}', dirs_exist_ok=True)
os.makedirs(f'{REPO_DIR}/output', exist_ok=True)
def _find_file(name):
    for root, _, files in os.walk(DS_DIR):
        if name in files: return os.path.join(root, name)
    return None
for fn in ('abt_v6_cached.parquet', 'v6_feature_manifest.json'):
    found = _find_file(fn)
    if found: shutil.copy(found, f'{REPO_DIR}/output/{fn}'); print('copied', fn)
    else: print('missing', fn)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print('ready')

In [ ]:
import os, json, pickle, time
import numpy as np, pandas as pd, optuna
from src.model_v2 import TwoStageForecaster, get_feature_columns_v2, encode_categoricals
from src.evaluation import compute_all_metrics

abt = pd.read_parquet('output/abt_v6_cached.parquet')
abt = encode_categoricals(abt)
feat_cols = get_feature_columns_v2(abt)
_p = abt['Период']
period = _p.dt.to_timestamp() if isinstance(_p.dtype, pd.PeriodDtype) else pd.to_datetime(_p)
abt['__per'] = period

# Use a 4-origin rolling CV for faster Optuna iteration.
ORIGINS = ['2025-10', '2025-11', '2025-12', '2026-01']

def run_trial(params):
    wapes = []
    for origin in ORIGINS:
        origin_ts = pd.Timestamp(origin)
        train_mask = abt['__per'] < origin_ts
        val_mask   = (abt['__per'] >= origin_ts - pd.DateOffset(months=3)) & (abt['__per'] < origin_ts)
        test_mask  = (abt['__per'] >= origin_ts) & (abt['__per'] < origin_ts + pd.DateOffset(months=1))
        df_tr = abt[train_mask]; df_val = abt[val_mask]; df_te = abt[test_mask]
        if df_te.empty: continue
        model = TwoStageForecaster(
            reg_params={**params, 'verbose': -1},
            reg_objective='pinball', reg_objective_kwargs={'alpha': 0.6},
            target_col='target_qty_imputed',
        )
        model.fit(df_tr, df_val, feat_cols, num_boost_round=800, early_stopping=50)
        p = model.predict(df_te)
        m = compute_all_metrics(df_te['target_qty'].to_numpy(), p)
        wapes.append(m['WAPE'])
    return float(np.mean(wapes) + 0.5 * np.std(wapes))

def objective(trial):
    params = {
        'objective': 'quantile',
        'alpha': 0.6,
        'num_leaves':        trial.suggest_int('num_leaves', 16, 255, log=True),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 0, 10),
        'min_data_in_leaf':  trial.suggest_int('min_data_in_leaf', 5, 200, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    return run_trial(params)

study = optuna.create_study(direction='minimize', study_name='v7', sampler=optuna.samplers.TPESampler(seed=42))
t0 = time.time()
study.optimize(objective, n_trials=120, show_progress_bar=False)
print('best', study.best_value, 'in', round(time.time()-t0), 's')
with open('/kaggle/working/v7_optuna_best_params.json', 'w') as f:
    json.dump({'best_score': study.best_value, 'best_params': study.best_params, 'n_trials': len(study.trials)}, f, indent=2)
with open('/kaggle/working/v7_optuna_study.pkl', 'wb') as f:
    pickle.dump(study, f)
top = sorted([(t.value, t.params) for t in study.trials if t.value is not None])[:10]
pd.DataFrame([{'rank': i+1, 'score': v, **p} for i,(v,p) in enumerate(top)]).to_csv('/kaggle/working/v7_optuna_top10.csv', index=False)
print('saved best_params:', study.best_params)